<a href="https://colab.research.google.com/github/rselvak6/opendde-colab/blob/main/opendde_monomer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenDDE on Colab

Requires a GPU runtime: **suggested: A100 high-ram**

This notebook: installs OpenDDE from source with `uv`, downloads model weights + inference data, runs a tiny smoke test, then runs your real prediction job.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d OpenDDE || git clone https://github.com/aurekaresearch/OpenDDE.git
%cd OpenDDE
!uv pip install --torch-backend cu126 -e '.[gpu]'
!uv pip install --group dev
!opendde doctor

downloading uv 0.11.31 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Cloning into 'OpenDDE'...
remote: Enumerating objects: 586, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 586 (delta 15), reused 12 (delta 11), pack-reused 559 (from 1)
Receiving objects: 100% (586/586), 20.84 MiB | 20.21 MiB/s, done.
Resolving deltas: 100% (266/266), done.
/content/OpenDDE
Using Python 3.12.13 environment at: /usr
Resolved 65 packages in 1.53s
Prepared 31 packages in 46.80s
Uninstalled 21 packages in 749ms
Installed 31 packages in 196ms
 + biopython==1.85
 + biotite==1.4.0
 + biotraj==1.2.2
 + cuequivariance==0.10.0
 + cuequivariance-ops-cu12==0.10.0
 + cuequivariance-ops-torch-cu12==0.10.0
 + cuequivariance-torch==0.10.0
 + ml-collections==1.1.0
 - numpy==2.0.2
 + numpy==2.4.1
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-

In [ ]:
# General-purpose checkpoint used by default with -n opendde_v1.
%env OPENDDE_ROOT_DIR=/content/data
!mkdir -p $OPENDDE_ROOT_DIR/checkpoint

env: OPENDDE_ROOT_DIR=/content/data


In [ ]:
# Download only what this notebook actually uses.
!apt-get -qq update && apt-get -qq install -y zstd
!bash /content/OpenDDE/scripts/download_opendde_data.sh --skip-search-database

In [ ]:
import os
os.environ["OPENDDE_FORCE_SAMPLE_DIFFUSION_AMP"] = "1"
os.environ["OPENDDE_FORCE_CONFIDENCE_AMP"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Must match the "name" field of the input JSON: OpenDDE writes results to
# ./output/<name>/, which every downstream cell reads, and the run cell below
# loads /content/OpenDDE/<PROJECT_NAME>.json. Pick a known input or type one.
PROJECT_NAME = 'egfr-rad51-monomer'  # @param ["egfr-rad51-monomer", "egfr-rad51-monomer-afatinib", "egfr-rad51-monomer-alflutinib", "egfr-rad51-monomer-bh30643", "egfr-rad51-monomer-erlotinib", "egfr-rad51-monomer-gefitinib", "egfr-rad51-monomer-lazertinib", "egfr-rad51-monomer-osimertinib"] {allow-input: true}

In [ ]:
# @title Template sweep setup: stage both supplied templates for PROJECT_NAME
# Set PROJECT_NAME in the cell above and run this; it prepares every template
# variant of that one compound in a single pass. The next cell then predicts
# all three. Nothing else needs changing per variant.
#
# The no-template variant keeps the bare PROJECT_NAME as its run name, so runs
# predicted before templates existed are reused rather than repeated.
# OpenDDE does NOT take a structure file as a template input. Its only template
# input is "templatesPath" on a proteinChain, which must point at an hmmsearch
# .a3m (or HHsearch .hhr) *hit* file; the structure itself is then looked up by
# the 4-character PDB ID in the hit name, under
# $OPENDDE_ROOT_DIR/search_database/mmcif/<pdb_id>.cif (falling back to a PDBe
# download for IDs it does not find locally). So to use a hand-built template we
# register it as if it were a PDB entry. This cell does that end to end.
#
# Four gates have to be satisfied, and three of them fail SILENTLY - the run
# just reports "Found 0 templates" and predicts as if --use_template were false:
#
#  1. The hit name must be 4 alphanumerics + "_" + chain
#     (template_parser.get_pdb_id_and_chain). We use IDs starting with a letter:
#     every one of the 219,912 real PDB IDs in release_date_cache.json starts
#     with a digit, so "egfa"/"egfi" can never collide with a real entry.
#  2. The ID must be present in $OPENDDE_ROOT_DIR/common/release_date_cache.json,
#     or TemplateHitFilter.prefilter drops the hit as "<id> date unknown".
#  3. The CIF's own min(_pdbx_audit_revision_history.revision_date) must be
#     <= 2021-09-30, the checkpoint's data cutoff, hardcoded as
#     max_template_date in infer_dataloader.py. A CIF with no revision date is
#     treated as 9999-12-31 and dropped. (The supplied files already carry
#     2021-09-30, so they pass with zero margin.)
#  4. Some monomer's _chem_comp.type must contain the lowercase substring
#     "peptide" (template_parser._get_protein_chains). The PDB writes
#     "L-peptide linking"; AlphaFold3 writes "L-PEPTIDE LINKING", which fails
#     that test and yields "No protein chains found". We normalize the casing on
#     the copy we stage, leaving the originals untouched.
#
# The .a3m needs less than you would expect: process() re-aligns the query to
# the CIF's own seqres with kalign and rebuilds the hit indices
# (_update_realigned_hit), so the alignment written here is used only for the
# prefilter's coverage ratio. We still emit a true alignment so the file is
# honest and stays correct if that ever changes.
TEMPLATE_DIR = "/content/templates"  # @param {type:"string"}

import glob
import io
import json
import os
import re
import shutil
import subprocess
import tempfile

ROOT = os.environ.get("OPENDDE_ROOT_DIR", "/content/data")
MMCIF_DIR = os.path.join(ROOT, "search_database", "mmcif")
TPL_CACHE_DIR = os.path.join(ROOT, "search_database", "template_cache")
DATE_CACHE = os.path.join(ROOT, "common", "release_date_cache.json")
A3M_DIR = "/content/template_hits"
INPUT_DIR = "/content/OpenDDE"

# filename substring -> 4-char registration ID. "idealinactive" is not a
# substring of "idealactive", so these two patterns stay disjoint.
TEMPLATES = {
    "active": {"id": "egfa", "match": "idealactive",
               "note": "EGFR kinase active supplied template"},
    "inactive": {"id": "egfi", "match": "idealinactive",
                 "note": "EGFR kinase inactive supplied template"},
}
CUTOFF = "2021-09-30"
MIN_TEMPLATE_COVERAGE = 0.5  # fraction of template residues matched to a chain


def _normalize_chem_comp_type(text):
    """Lowercase "PEPTIDE" inside _chem_comp.type values (PDB casing).

    Data-driven: only the exact quoted strings that actually appear in this
    file's _chem_comp.type column are rewritten, so nothing else is touched.
    """
    from Bio.PDB.MMCIF2Dict import MMCIF2Dict

    types = set(MMCIF2Dict(io.StringIO(text)).get("_chem_comp.type", []))
    changed = []
    for t in types:
        if "PEPTIDE" in t.upper() and "peptide" not in t:
            fixed = re.sub("PEPTIDE", "peptide", t, flags=re.I)
            for q in ('"%s"' % t, "'%s'" % t):
                if q in text:
                    text = text.replace(q, q[0] + fixed + q[0])
            changed.append("%s -> %s" % (t, fixed))
    return text, changed


def _parse_template_cif(text):
    """Returns (seqres_one_letter, author_chain_id, release_date).

    Mirrors template_parser.TemplateParser.parse so a file that fails here
    would have failed inside OpenDDE too.
    """
    import collections

    from Bio import PDB
    from Bio.Data import PDBData

    p = PDB.MMCIFParser(QUIET=True)
    p.get_structure("", io.StringIO(text))
    info = dict(p._mmcif_dict)
    for k, v in list(info.items()):
        if not isinstance(v, list):
            info[k] = [v]

    def loop(prefix):
        cols = [k for k in info if k.startswith(prefix)]
        if not cols:
            return []
        return [dict(zip(cols, row)) for row in zip(*(info[c] for c in cols))]

    dates = info.get("_pdbx_audit_revision_history.revision_date") or []
    release = min(dates) if dates else ""

    polymers = collections.defaultdict(list)
    for e in loop("_entity_poly_seq."):
        polymers[e["_entity_poly_seq.entity_id"]].append(e["_entity_poly_seq.mon_id"])
    chem = {e["_chem_comp.id"]: e for e in loop("_chem_comp.")}
    ent2chain = collections.defaultdict(list)
    for a in loop("_struct_asym."):
        ent2chain[a["_struct_asym.entity_id"]].append(a["_struct_asym.id"])

    valid = {}
    for eid, mons in polymers.items():
        if any("peptide" in chem.get(m, {}).get("_chem_comp.type", "") for m in mons):
            for cid in ent2chain[eid]:
                valid[cid] = mons
    if not valid:
        raise RuntimeError("no protein chains (gate 4) - _chem_comp.type casing?")

    label2auth = {}
    for lab, auth, mdl in zip(info["_atom_site.label_asym_id"],
                              info["_atom_site.auth_asym_id"],
                              info["_atom_site.pdbx_PDB_model_num"]):
        if mdl == "1":
            label2auth[lab] = auth
    for cid, mons in valid.items():
        if cid in label2auth:
            seq = "".join(PDBData.protein_letters_3to1.get(m, "X") for m in mons)
            seq = "".join(c if len(c) == 1 else "X" for c in seq)
            return seq, label2auth[cid], release
    raise RuntimeError("no protein chain present in model 1")


def _align_blocks(query, tseq):
    from Bio import Align
    from Bio.Align import substitution_matrices

    aligner = Align.PairwiseAligner()
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    aligner.open_gap_score = -11
    aligner.extend_gap_score = -1
    aligner.mode = "local"
    aln = aligner.align(query, tseq)[0]
    return [(tuple(q), tuple(t)) for q, t in zip(*aln.aligned)]


def _coverage(query, tseq, blocks):
    ident = sum(
        1
        for (qs, qe), (ts, _) in blocks
        for k in range(qe - qs)
        if query[qs + k] == tseq[ts + k]
    )
    return ident / len(tseq)


def _a3m_text(query, tseq, tpl_id, chain, note, blocks):
    """One hmmsearch-style a3m hit: query columns upper/'-', insertions lower."""
    col = ["-"] * len(query)
    ins = [""] * len(query)
    prev_q_end = prev_t_end = None
    for (qs, qe), (ts, te) in blocks:
        if prev_t_end is not None and ts > prev_t_end:
            ins[prev_q_end - 1] += tseq[prev_t_end:ts].lower()
        for k in range(qe - qs):
            col[qs + k] = tseq[ts + k].upper()
        prev_q_end, prev_t_end = qe, te
    start = blocks[0][1][0] + 1
    end = blocks[-1][1][1]
    hit = "".join(col[i] + ins[i] for i in range(len(query)))
    desc = ">%s_%s/%d-%d mol:protein length:%d  %s" % (
        tpl_id, chain, start, end, len(tseq), note)
    return desc + "\n" + hit + "\n"


def _patch_date_cache(ids):
    if not os.path.exists(DATE_CACHE):
        raise RuntimeError(
            "missing %s - run the data-download cell first (it is part of the "
            "'common' files, not the skipped search database)." % DATE_CACHE)
    with open(DATE_CACHE) as f:
        cache = json.load(f)
    added = [i for i in ids if cache.get(i, {}).get("release_date") != CUTOFF]
    if added:
        for i in added:
            cache[i] = {"release_date": CUTOFF}
        tmp = DATE_CACHE + ".tmp"
        with open(tmp, "w") as f:
            json.dump(cache, f)
        os.replace(tmp, DATE_CACHE)
    return added, len(cache)


def _check_kalign():
    exe = shutil.which("kalign")
    if exe is None:
        print("  kalign not found - installing (needed to align query->template)")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(["apt-get", "-qq", "install", "-y", "kalign"], check=False)
        exe = shutil.which("kalign")
    if exe is None:
        raise RuntimeError("kalign still unavailable; templates cannot be used")
    # Smoke-test the exact CLI OpenDDE calls (kalign 3.x: -i / -o / -format).
    with tempfile.TemporaryDirectory() as d:
        fi, fo = os.path.join(d, "i.fa"), os.path.join(d, "o.a3m")
        with open(fi, "w") as f:
            f.write(">a\nACDEFGHIKL\n>b\nACDEFGHIKL\n")
        r = subprocess.run([exe, "-i", fi, "-o", fo, "-format", "fasta"],
                           capture_output=True, text=True)
        if r.returncode or not os.path.exists(fo):
            raise RuntimeError(
                "kalign at %s does not accept OpenDDE's '-i/-o/-format fasta' "
                "CLI (needs kalign 3.x).\nstderr: %s" % (exe, r.stderr[:400]))
    print("  kalign OK: %s" % exe)
    return exe


def stage_template(which, base_name, template_dir, input_dir=INPUT_DIR,
                   verify_kalign=True):
    """Stages one template and returns (input_json, run_name)."""
    spec = TEMPLATES[which]
    tpl_id = spec["id"]
    print("Staging %s template as PDB-like entry '%s'" % (which, tpl_id))
    if verify_kalign:
        _check_kalign()

    cands = [p for p in sorted(glob.glob(os.path.join(template_dir, "*.cif")))
             if spec["match"] in os.path.basename(p).lower()]
    if len(cands) != 1:
        raise RuntimeError(
            "expected exactly 1 *.cif matching '%s' in %s, found %d: %s\n"
            "Upload the two template CIFs to %s first."
            % (spec["match"], template_dir, len(cands),
               [os.path.basename(c) for c in cands], template_dir))
    src = cands[0]
    print("  source: %s" % os.path.basename(src))

    with open(src, encoding="utf-8") as f:
        text = f.read()
    text, changed = _normalize_chem_comp_type(text)
    for c in changed:
        print("  _chem_comp.type normalized: %s" % c)
    tseq, chain, release = _parse_template_cif(text)
    print("  chain %s, %d residues, revision_date %s" % (chain, len(tseq), release))
    if not release or release > CUTOFF:
        raise RuntimeError(
            "release date %r is missing or after the %s cutoff; the hit would be "
            "dropped silently." % (release, CUTOFF))

    os.makedirs(MMCIF_DIR, exist_ok=True)
    os.makedirs(A3M_DIR, exist_ok=True)
    staged = os.path.join(MMCIF_DIR, tpl_id + ".cif")
    with open(staged, "w", encoding="utf-8") as f:
        f.write(text)
    print("  staged CIF: %s" % staged)

    # A parsed-CIF pickle from an earlier staging of the same ID would shadow
    # the file just written.
    for p in glob.glob(os.path.join(TPL_CACHE_DIR, "%s_*.pkl" % tpl_id)):
        os.remove(p)
        print("  cleared stale parse cache: %s" % os.path.basename(p))

    added, total = _patch_date_cache([tpl_id])
    print("  release_date_cache.json: %s (%d entries)"
          % ("added " + tpl_id if added else tpl_id + " already present", total))

    base_json = os.path.join(input_dir, base_name + ".json")
    with open(base_json) as f:
        jobs = json.load(f)
    if len(jobs) != 1:
        raise RuntimeError("expected a single job in %s, got %d"
                           % (base_json, len(jobs)))
    job = jobs[0]

    attached = skipped = 0
    for idx, ent in enumerate(job["sequences"]):
        if "proteinChain" not in ent:
            continue
        chain_seq = ent["proteinChain"]["sequence"]
        blocks = _align_blocks(chain_seq, tseq)
        cov = _coverage(chain_seq, tseq, blocks)
        ids = ent["proteinChain"].get("id")
        if cov < MIN_TEMPLATE_COVERAGE:
            skipped += 1
            print("  chain(s) %s: coverage %.2f < %.2f -> no template (correct "
                  "for non-EGFR chains)" % (ids, cov, MIN_TEMPLATE_COVERAGE))
            continue
        a3m = os.path.join(A3M_DIR, "%s_e%d.a3m" % (tpl_id, idx))
        with open(a3m, "w") as f:
            f.write(_a3m_text(chain_seq, tseq, tpl_id, chain, spec["note"], blocks))
        ent["proteinChain"]["templatesPath"] = a3m
        attached += 1
        print("  chain(s) %s: coverage %.2f -> %s" % (ids, cov, a3m))
    if not attached:
        raise RuntimeError("no protein chain matched the template; nothing to do")

    run_name = "%s-t%s" % (base_name, which)
    job["name"] = run_name
    out_json = os.path.join(input_dir, run_name + ".json")
    with open(out_json, "w") as f:
        json.dump(jobs, f, indent=2)
    print("  wrote %s (%d chain(s) templated, %d skipped)"
          % (out_json, attached, skipped))
    return out_json, run_name

# ---------------------------------------------------------------- sweep
# One entry per template variant of this compound. "none" deliberately keeps
# the bare project name: its output directory is the one earlier no-template
# runs already wrote to, so the pred cell below can skip it instead of
# spending a GPU hour reproducing it.
BASE_NAME = PROJECT_NAME
RUNS = [{
    "template": "none",
    "run_name": BASE_NAME,
    "input_json": os.path.join(INPUT_DIR, BASE_NAME + ".json"),
    "use_template": "false",
}]
for _i, _which in enumerate(("active", "inactive")):
    print()
    _json, _run = stage_template(_which, BASE_NAME, TEMPLATE_DIR,
                                 verify_kalign=(_i == 0))
    RUNS.append({"template": _which, "run_name": _run,
                 "input_json": _json, "use_template": "true"})

RUNS_BY_TEMPLATE = {r["template"]: r for r in RUNS}
# PROJECT_NAME is left at the base name here: the sweep produces three runs, so
# the analysis cells are pointed at one of them by the selector cell further
# down rather than by this cell.
print()
print("=" * 72)
print("%d variants staged for %s" % (len(RUNS), BASE_NAME))
for r in RUNS:
    print("  %-9s use_template=%-5s ./output/%s" %
          (r["template"], r["use_template"], r["run_name"]))
print("=" * 72)


In [ ]:
# @title Predict every template variant of PROJECT_NAME
# One `opendde pred` per entry in RUNS. The template is selected per run by
# --use_template plus the generated input JSON (whose proteinChain carries
# templatesPath), so a single hard-coded pred line cannot cover the sweep.
#
# A variant whose output directory already holds .cif files is skipped, which
# makes the cell safe to re-run after a crash or an OOM on one variant. Tick
# FORCE_RERUN to redo them anyway.
FORCE_RERUN = False  # @param {type:"boolean"}

import glob
import os
import subprocess
import threading
import time

if "RUNS" not in globals():
    raise RuntimeError("run the template sweep cell above first")

# Per-variant GPU utilisation, read by the "Check GPU utilisation" cell.
# Colab runs one cell at a time, so utilisation during a prediction cannot
# be observed from another cell - it has to be sampled from inside this one.
GPU_PEAKS = globals().get("GPU_PEAKS", {})


def _sample_gpus(stop_event, peaks, period=5.0):
    """Poll nvidia-smi until stop_event is set. Never raises into the run."""
    while not stop_event.is_set():
        try:
            out = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=index,utilization.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=30).stdout
            for line in out.strip().splitlines():
                idx, util, mem = (c.strip() for c in line.split(","))
                p = peaks.setdefault(int(idx), {"max_util": 0, "sum_util": 0,
                                                "n": 0, "max_mem": 0})
                p["max_util"] = max(p["max_util"], int(util))
                p["max_mem"] = max(p["max_mem"], int(mem))
                p["sum_util"] += int(util)
                p["n"] += 1
        except Exception:
            return  # no nvidia-smi, or it misbehaved: just stop sampling
        stop_event.wait(period)


PRED_FLAGS = ["-o", "./output", "-n", "opendde_v1", "--use_msa", "true",
              "--use_rna_msa", "false", "--sample", "1", "--step", "200",
              "--cycle", "10", "--need_atom_confidence", "true", "--dtype", "bf16"]

_summary = []
for r in RUNS:
    out_dir = os.path.join("./output", r["run_name"])
    existing = glob.glob(os.path.join(out_dir, "**", "*.cif"), recursive=True)
    label = "template=%s -> ./output/%s" % (r["template"], r["run_name"])
    if existing and not FORCE_RERUN:
        print("== SKIP  %s (%d .cif already there)" % (label, len(existing)))
        _summary.append((r["template"], r["run_name"], "skipped", len(existing)))
        continue
    cmd = (["opendde", "pred", "-i", r["input_json"],
            "--use_template", r["use_template"]] + PRED_FLAGS)
    print()
    print("== RUN   %s" % label)
    print("   %s" % " ".join(cmd))
    print()
    _peaks = {}
    _stop = threading.Event()
    _sampler = threading.Thread(target=_sample_gpus, args=(_stop, _peaks),
                                daemon=True)
    _sampler.start()
    _t0 = time.time()
    try:
        code = subprocess.run(cmd).returncode
    finally:
        _stop.set()
        _sampler.join(timeout=10)
    GPU_PEAKS[r["run_name"]] = _peaks
    print("   %s in %.1f min%s" % (
        "finished" if code == 0 else "failed", (time.time() - _t0) / 60,
        "".join(", GPU %d peak %d%% / %d MiB"
                % (i, _peaks[i]["max_util"], _peaks[i]["max_mem"])
                for i in sorted(_peaks))))
    n = len(glob.glob(os.path.join(out_dir, "**", "*.cif"), recursive=True))
    _summary.append((r["template"], r["run_name"],
                     "ok" if code == 0 else "FAILED rc=%d" % code, n))
    if code != 0:
        # Keep going: an OOM on one variant should not cost the other two.
        print("!! %s failed (exit %d) - continuing with the remaining variants"
              % (label, code))

print()
print("=" * 72)
for _t, _name, _state, _n in _summary:
    _p = GPU_PEAKS.get(_name) or {}
    _gpu = " | ".join("GPU %d mean %d%% peak %d%%"
                      % (i, _p[i]["sum_util"] // max(1, _p[i]["n"]),
                         _p[i]["max_util"]) for i in sorted(_p))
    print("  %-9s %-40s %-12s %d .cif  %s" % (_t, _name, _state, _n, _gpu))
print("=" * 72)
if any(s[2].startswith("FAILED") for s in _summary):
    print("Re-running this cell retries only the failed variants.")


## Post-processing: visualize structures + rank by ipSAE

Requires the `pred` runs above to have used `--need_atom_confidence true` (adds `*_full_data_sample_*.json` per model, with the PAE matrix and per-atom pLDDT).

In [ ]:
# @title Which template variant should the cells below read?
# The sweep writes three output directories; every analysis cell below reads
# ./output/<PROJECT_NAME>, so this is where you choose between them. Re-run
# this cell and the ones after it to switch variants.
ANALYZE_TEMPLATE = "none"  # @param ["none", "active", "inactive"]

if "RUNS_BY_TEMPLATE" not in globals():
    raise RuntimeError("run the template sweep cell above first")
PROJECT_NAME = RUNS_BY_TEMPLATE[ANALYZE_TEMPLATE]["run_name"]
print("analysis cells below will read ./output/%s" % PROJECT_NAME)


In [ ]:
# @title Shared helpers: locate each model's .cif plus its matching per-sample JSON
# sidecars, following OpenDDE's dumper.py naming convention:
#   {name}_sample_{rank}.cif
#   {name}_summary_confidence_sample_{rank}.json   (always written)
#   {name}_full_data_sample_{rank}.json             (only with --need_atom_confidence true)
import glob
import os
import re

_SAMPLE_RE = re.compile(r"(.+)_sample_(\d+)\.cif$")


def find_all_cifs(output_dir="./output/{}".format(PROJECT_NAME)):
    return sorted(glob.glob(os.path.join(output_dir, "**", "*.cif"), recursive=True))


def _sample_stem_rank(cif_path):
    m = _SAMPLE_RE.match(os.path.basename(cif_path))
    return m.groups() if m else (None, None)


def find_full_data_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_full_data_sample_{rank}.json")
    return p if os.path.exists(p) else None


def find_summary_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_summary_confidence_sample_{rank}.json")
    return p if os.path.exists(p) else None


def model_label(cif_path):
    # Short "seed N, sample M" label instead of the full relative path.
    _, rank = _sample_stem_rank(cif_path)
    seed_match = re.search(r"seed_(\d+)", cif_path)
    seed = seed_match.group(1) if seed_match else "?"
    return f"seed {seed}, sample {rank}" if rank is not None else os.path.basename(cif_path)


cifs = find_all_cifs()
print(f"Found {len(cifs)} predicted structure(s) under ./output/{PROJECT_NAME}")

Found 0 predicted structure(s) under ./output/egfr-full


In [ ]:
#@title Visualize predicted structures + PAE (py2Dmol)
!pip install -q py3Dmol py2Dmol

import json

import numpy as np
import py2Dmol

load_as_frames = True  # @param {type:"boolean"}
viewer_size = 400


def load_pae(cif_path):
    # OpenDDE's full_data.json stores the PAE matrix as "token_pair_pae"
    # (AlphaFold3's equivalent full_data.json key is "pae").
    fd_path = find_full_data_json(cif_path)
    if fd_path is None:
        return None
    with open(fd_path) as f:
        data = json.load(f)
    pae = data.get("token_pair_pae")
    return np.asarray(pae, dtype=float) if pae is not None else None


if not cifs:
    print("No .cif files found under ./output yet - run a prediction first.")
else:
    viewer = py2Dmol.view(size=(viewer_size, viewer_size), pae=True, autoplay=load_as_frames)
    for i, cif in enumerate(cifs, start=1):
        pae = load_pae(cif)
        if pae is None:
            print(f"No PAE data for {cif} (re-run pred with --need_atom_confidence true)")
        name = "models" if load_as_frames else os.path.basename(cif).replace(".cif", "")
        viewer.add_pdb(cif, name=name, paes=pae)
    viewer.show()

No .cif files found under ./output yet - run a prediction first.


## Utilities

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("{}".format(PROJECT_NAME), "zip", "./output/{}".format(PROJECT_NAME))
files.download("{}.zip".format(PROJECT_NAME))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import glob, os

for pattern in ("*.txt", "*.pml"):
    for f in glob.glob(f"./output/**/{pattern}", recursive=True):
        os.remove(f)

In [ ]:
# @title Check GPU utilisation
# What "all the GPUs" can mean here, and what this cell can actually tell you.
#
# `opendde pred`, as the sweep cell above invokes it, is a SINGLE-GPU run no
# matter how many devices the runtime has. OpenDDE takes its world size from the
# WORLD_SIZE environment variable (opendde/utils/distributed.py), which only
# torchrun sets, so a bare `opendde pred` always has world_size=1. The run log
# says exactly that: "world size: 1", "CUDA_VISIBLE_DEVICES: [0]",
# "size_dp=1, size_cp=1, mesh=(1, 1)".
#
# Using more than one GPU means the Fold-CP context-parallel path under
# torchrun, where --nproc_per_node must equal --foldcp_size_cp:
#
#   CUDA_VISIBLE_DEVICES=0,1 torchrun --standalone --nproc_per_node 2 \
#     -m runner.batch_inference pred -i <input.json> -o ./output -n opendde_v1 \
#     --use_msa true --use_template <true|false> --use_rna_msa false \
#     --sample 1 --step 200 --cycle 10 --need_atom_confidence true \
#     --dtype bf16 --triatt_kernel torch --trimul_kernel torch \
#     --foldcp_mode distributed --foldcp_size_dp 1 --foldcp_size_cp 2
#
# Mind the kernels: Fold-CP does not support cuEquivariance, so distributed mode
# forces the PyTorch triangle kernels and an explicit cueq request fails before
# the model loads. This run uses cueq ("Resolved triangle kernels from auto to
# multiplicative=cuequivariance, attention=cuequivariance"), which is faster per
# device, so a 2-way CP mesh is not automatically a win - measure it on one
# compound before converting the sweep. Fold-CP's value here is headroom, not
# throughput: it splits the token-pair tensors that caused the earlier OOM.
#
# On a one-GPU runtime, "properly utilised" just means that device stayed busy
# with sane memory headroom, which is what the sampling below reports.
SAMPLE_SECONDS = 6  # @param {type:"integer"}

import os
import subprocess
import time

QUERY = "index,name,memory.total,memory.used,utilization.gpu,temperature.gpu"


def _smi(query, extra=None):
    # The "--query-x=fields" form is the one NVIDIA documents; nvidia-smi also
    # accepts a space, but keep to the documented spelling.
    cmd = ["nvidia-smi", "--query-%s=%s" % (extra or "gpu", query),
           "--format=csv,noheader,nounits"]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
        return None, str(exc)
    if r.returncode:
        return None, r.stderr.strip()
    return [[c.strip() for c in ln.split(",")] for ln in r.stdout.strip().splitlines()], None


rows, err = _smi(QUERY)
if rows is None:
    print("nvidia-smi unavailable (%s) - no GPU attached? "
          "Runtime > Change runtime type > GPU." % err)
else:
    print("%d GPU(s) visible to nvidia-smi:" % len(rows))
    print("  %-3s %-26s %-18s %-7s %s" % ("idx", "name", "memory used/total", "util", "temp"))
    for idx, name, total, used, util, temp in rows:
        print("  %-3s %-26s %6s / %6s MiB %5s%% %s C"
              % (idx, name[:26], used, total, util, temp))

    print()
    print("CUDA_VISIBLE_DEVICES=%r" % os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
    try:
        import torch
        print("torch sees %d device(s); built for CUDA %s"
              % (torch.cuda.device_count(), torch.version.cuda))
    except Exception as exc:
        print("torch unavailable: %s" % exc)

    apps, aerr = _smi("gpu_uuid,pid,used_memory", extra="compute-apps")
    print()
    if apps:
        print("%d compute process(es) on the GPU(s) right now:" % len(apps))
        for a in apps:
            print("   ", ", ".join(a))
    else:
        print("No compute processes running right now (%s)."
              % ("expected if the sweep has finished" if aerr is None else aerr))

    # Sample, so an idle-looking instant reading is not mistaken for an idle GPU.
    print()
    print("Sampling utilisation for %ds ..." % SAMPLE_SECONDS)
    seen = {}
    deadline = time.time() + max(1, SAMPLE_SECONDS)
    while time.time() < deadline:
        s, _ = _smi("index,utilization.gpu,memory.used")
        for r_ in s or []:
            try:
                i, u, m = int(r_[0]), int(r_[1]), int(r_[2])
            except (ValueError, IndexError):
                continue
            d = seen.setdefault(i, {"max_util": 0, "sum": 0, "n": 0, "max_mem": 0})
            d["max_util"] = max(d["max_util"], u)
            d["max_mem"] = max(d["max_mem"], m)
            d["sum"] += u
            d["n"] += 1
        time.sleep(0.5)
    for i in sorted(seen):
        d = seen[i]
        print("  GPU %d: peak %d%%, mean %d%%, peak %d MiB used"
              % (i, d["max_util"], d["sum"] // max(1, d["n"]), d["max_mem"]))

    # What the sweep itself recorded, if the pred cell has run in this session.
    if "GPU_PEAKS" in globals() and GPU_PEAKS:
        print()
        print("Recorded while each prediction was running:")
        for run_name, peaks in GPU_PEAKS.items():
            if not peaks:
                print("  %-44s (not sampled)" % run_name)
                continue
            for i in sorted(peaks):
                p = peaks[i]
                print("  %-44s GPU %d: peak %d%%, mean %d%%, peak %d MiB"
                      % (run_name, i, p["max_util"],
                         p["sum_util"] // max(1, p["n"]), p["max_mem"]))
    else:
        print()
        print("No per-run samples in this session yet (GPU_PEAKS is filled in by "
              "the prediction cell as each variant runs).")

    print()
    n_gpu = len(rows)
    if n_gpu > 1:
        print("NOTE: %d GPUs are attached but the sweep uses one. See the comment at "
              "the top of this cell for the torchrun/Fold-CP invocation, and the "
              "kernel caveat before switching." % n_gpu)
    else:
        print("One GPU attached, which is what the sweep uses - nothing is being "
              "left idle.")

In [ ]:
# @title Disconnect and delete the runtime
# IRREVERSIBLE. Deleting the runtime wipes this machine's disk: ./output, every
# .zip still sitting here, the staged templates, and the ~90 GB of downloaded
# model/common data. Nothing is recoverable afterwards. The upside is that the
# GPU is released immediately instead of idling against your quota.
#
# The sweep writes one output directory PER TEMPLATE VARIANT while the download
# cell above zips only the variant currently selected, so the easiest way to
# lose work here is to download one variant and delete the runtime with the
# other two still on disk. This cell therefore audits every run first and
# refuses to delete anything that has .cif files but no .zip beside them.
#
# A .zip on disk only proves you zipped it, not that the browser finished
# saving it - check your own Downloads folder before ticking the box.
I_HAVE_DOWNLOADED_THE_RESULTS = False  # @param {type:"boolean"}
DELETE_EVEN_IF_ZIPS_ARE_MISSING = False  # @param {type:"boolean"}

import glob
import os

# The install cell does `%cd OpenDDE`, so make_archive in the download cell
# writes to /content/OpenDDE; check the plausible places regardless.
ZIP_DIRS = [os.getcwd(), "/content", "/content/OpenDDE"]

if "RUNS" in globals() and RUNS:
    run_names = [r["run_name"] for r in RUNS]
else:
    # Fall back to whatever is on disk, so this works without the sweep cell.
    run_names = sorted(os.path.basename(p.rstrip("/\\"))
                       for p in glob.glob("./output/*") if os.path.isdir(p))

audit = []
for name in run_names:
    n_cif = len(glob.glob(os.path.join("./output", name, "**", "*.cif"), recursive=True))
    zpath = next((os.path.join(d, name + ".zip") for d in ZIP_DIRS
                  if os.path.isfile(os.path.join(d, name + ".zip"))), None)
    audit.append((name, n_cif, zpath))

print("%-44s %-9s %s" % ("run", ".cif", "zip on disk"))
print("-" * 92)
for name, n_cif, zpath in audit:
    size = "%.1f MB" % (os.path.getsize(zpath) / 1e6) if zpath else ""
    print("%-44s %-9d %s %s" % (name, n_cif, zpath or "- none -", size))
print("-" * 92)

at_risk = [name for name, n_cif, zpath in audit if n_cif and not zpath]
empty = [name for name, n_cif, _ in audit if not n_cif]
if empty:
    print("No .cif files (nothing to lose): %s" % ", ".join(empty))

if not audit:
    print()
    print("No runs found under ./output - nothing to protect.")

if not I_HAVE_DOWNLOADED_THE_RESULTS:
    print()
    print("Nothing deleted. The runtime is still running and still consuming quota.")
    print("Download what you need, then tick I_HAVE_DOWNLOADED_THE_RESULTS and "
          "re-run this cell.")
    if at_risk:
        print("Not yet zipped: %s" % ", ".join(at_risk))
elif at_risk and not DELETE_EVEN_IF_ZIPS_ARE_MISSING:
    raise RuntimeError(
        "Refusing to delete the runtime: these runs have .cif files but no .zip "
        "on disk, so their results exist nowhere else:\n  "
        + "\n  ".join(at_risk)
        + "\n\nSelect each one in the analysis selector cell and run the download "
          "cell for it, or tick DELETE_EVEN_IF_ZIPS_ARE_MISSING to discard them "
          "on purpose.")
else:
    if at_risk:
        print()
        print("Discarding un-zipped results on purpose: %s" % ", ".join(at_risk))
    print()
    print("Unassigning the runtime now. The notebook will disconnect; its disk, "
          "including ./output, is gone.")
    try:
        from google.colab import runtime
    except ImportError as exc:
        raise RuntimeError(
            "google.colab.runtime is unavailable (%s) - this cell only does "
            "anything on Colab. Stop the kernel yourself elsewhere." % exc)
    runtime.unassign()